# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zigs166/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 (AI-driven SEO / content refresh performance gains):
- Label origin: Where is the refresh or performance outcome label defined from (for example, observed traffic/ranking changes over a fixed window vs. editorial flags), and could noisy or seasonal metrics influence which pages look improved?
- Validation design check: Does the evaluation split separate training and test rows cleanly across time and entities so shared context or overlapping windows do not make the result look stronger than it is?

Finding 2 (Search visibility / CTR & ranking signal relationships):
- Label origin: How are target/priority labels constructed from search analytics signals, and are there proxy definitions or thresholds that could be sensitive to SERP layout changes or missing history?
- Validation design check: Does the validation setup test generalization to unseen periods or groups rather than only random/in-sample slices?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import json
import numpy as np
import pandas as pd

# Optional HF_TOKEN check from Colab Secrets (never hardcoded)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
except Exception:
    pass

# Load dataset from repo/local/GitHub fallback
csv_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in csv_candidates if p.startswith("http") or os.path.exists(p)), None)
df_audit_raw = pd.read_csv(data_path)
print("Loaded dataset from:", data_path)
print("Row count:", len(df_audit_raw), "| Columns:", list(df_audit_raw.columns))

audit_notes = [
    {
        "finding": "AI-driven SEO & content refresh impact",
        "label_question": "How is the outcome label defined and isolated from noise/seasonality?",
        "validation_question": "Does the split prevent time/group overlap between train and evaluation sets?",
    },
    {
        "finding": "Search visibility & CTR/position relationships",
        "label_question": "Are thresholds/proxies stable across SERP shifts and varying data history?",
        "validation_question": "Does validation measure out-of-group / out-of-time generalization?",
    },
]
display(pd.DataFrame(audit_notes))

Loaded dataset from: https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv
Row count: 30000 | Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,finding,label_question,validation_question
0,AI-driven SEO & content refresh impact,How is the outcome label defined and isolated ...,Does the split prevent time/group overlap betw...
1,Search visibility & CTR/position relationships,Are thresholds/proxies stable across SERP shif...,Does validation measure out-of-group / out-of-...


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before vs. After Split Comparison: We compare our Week-5 model under a standard train/test split ("Before") against an honest grouped split using GroupKFold ("After") on deterministic content/row groups so related rows stay together during evaluation, alongside a few sample error rows.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

num_cols = df_audit_raw.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    df_audit_raw["feat_base"] = np.arange(1, len(df_audit_raw) + 1, dtype=float)
    num_cols = ["feat_base"]

s_a = df_audit_raw[num_cols[0]].astype(float).fillna(0.0)
s_b = df_audit_raw[num_cols[1]].astype(float).fillna(10.0) if len(num_cols) > 1 else pd.Series(10.0, index=df_audit_raw.index)
s_c = df_audit_raw[num_cols[2]].astype(float).fillna(0.0) if len(num_cols) > 2 else (s_a * 0.03)

X_aud = pd.DataFrame({
    "f_volume": s_a,
    "f_position": s_b,
    "f_clicks": s_c,
    "f_ctr_proxy": s_c / (s_a + 1.0),
    "f_inv_pos": 1.0 / (s_b.abs() + 1.0),
}, index=df_audit_raw.index)

priority_signal = (1.0 - (X_aud["f_ctr_proxy"] - X_aud["f_ctr_proxy"].min()) / (X_aud["f_ctr_proxy"].max() - X_aud["f_ctr_proxy"].min() + 1e-9)) * 0.6 + (
    (X_aud["f_inv_pos"] - X_aud["f_inv_pos"].min()) / (X_aud["f_inv_pos"].max() - X_aud["f_inv_pos"].min() + 1e-9)
) * 0.4
y_aud = (priority_signal >= priority_signal.median()).astype(int)

# BEFORE: standard holdout split
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_aud, y_aud, test_size=0.30, random_state=42, stratify=y_aud if y_aud.nunique() > 1 else None
)
pipe_b = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
pipe_b.fit(X_tr_b, y_tr_b)
prob_b = pipe_b.predict_proba(X_te_b)[:, 1]
pred_b = (prob_b >= 0.5).astype(int)
auc_b = roc_auc_score(y_te_b, prob_b) if y_te_b.nunique() > 1 else np.nan
acc_b = accuracy_score(y_te_b, pred_b)

# AFTER: honest grouped split (GroupKFold)
group_col = next((c for c in df_audit_raw.columns if "group" in c.lower() or "client" in c.lower() or "site" in c.lower()), None)
if group_col is not None:
    groups = df_audit_raw[group_col].astype(str)
else:
    # Deterministic bucket groups when no explicit client/group column exists
    groups = pd.Series([f"grp_{i % 25}" for i in range(len(df_audit_raw))], index=df_audit_raw.index)

n_splits = min(5, groups.nunique())
gkf = GroupKFold(n_splits=n_splits)
aucs_after, accs_after = [], []
sample_errors = None

for tr_idx, te_idx in gkf.split(X_aud, y_aud, groups=groups):
    X_tr_g, X_te_g = X_aud.iloc[tr_idx], X_aud.iloc[te_idx]
    y_tr_g, y_te_g = y_aud.iloc[tr_idx], y_aud.iloc[te_idx]
    pipe_g = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
    pipe_g.fit(X_tr_g, y_tr_g)
    prob_g = pipe_g.predict_proba(X_te_g)[:, 1]
    pred_g = (prob_g >= 0.5).astype(int)
    if y_te_g.nunique() > 1:
        aucs_after.append(roc_auc_score(y_te_g, prob_g))
    accs_after.append(accuracy_score(y_te_g, pred_g))
    if sample_errors is None:
        err_tmp = X_te_g.copy()
        err_tmp["y_true"] = y_te_g.values
        err_tmp["y_pred"] = pred_g
        err_tmp["pred_prob"] = np.round(prob_g, 4)
        sample_errors = err_tmp[err_tmp["y_true"] != err_tmp["y_pred"]].head(5)

before_after_df = pd.DataFrame([
    {
        "split_stage": "Before (Standard Split)",
        "roc_auc": round(float(auc_b), 4) if pd.notna(auc_b) else None,
        "accuracy": round(float(acc_b), 4),
    },
    {
        "split_stage": f"After (GroupKFold k={n_splits})",
        "roc_auc": round(float(np.mean(aucs_after)), 4) if len(aucs_after) > 0 else None,
        "accuracy": round(float(np.mean(accs_after)), 4),
    },
])
display(before_after_df)

print("\nSample failure / error examples under grouped validation:")
display(sample_errors)


,split_stage,roc_auc,accuracy
0,Before (Standard Split),0.9999,0.9646
1,After (GroupKFold k=5),0.9997,0.9696



Sample failure / error examples under grouped validation:


,f_volume,f_position,f_clicks,f_ctr_proxy,f_inv_pos,y_true,y_pred,pred_prob
87,210.0,0.02,0.00,0.000000,0.980392,0,1,0.5740
157,40.0,0.02,0.12,0.002927,0.980392,0,1,0.5153
178,50.0,0.02,0.00,0.000000,0.980392,0,1,0.6144
404,10.0,0.02,0.00,0.000000,0.980392,0,1,0.6242
635,10.0,0.01,0.00,0.000000,0.990099,0,1,0.8481


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage Audit: We check our final feature set to verify that no target/outcome columns, future-window metrics, or forbidden identifier/flag fields are included in the model inputs.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
final_features = list(X_aud.columns)
forbidden_terms = ["target", "label", "future", "leak", "outcome", "actual", "url", "client", "query"]
flagged_features = [col for col in final_features if any(term in col.lower() for term in forbidden_terms)]

assert len(flagged_features) == 0, f"Potential leakage/forbidden columns found: {flagged_features}"
print("Leakage audit passed on final features:", final_features)


Leakage audit passed on final features: ['f_volume', 'f_position', 'f_clicks', 'f_ctr_proxy', 'f_inv_pos']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Bold Claim (Before): "This model proves which pages will gain traffic after a refresh and automatically decides what content to rewrite."
Safe Rewrite (After): "Based on measured historical signals in this observed slice, the model provides a directional priority ranking as a decision-support aid for human content-refresh review."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs("work/outputs", exist_ok=True)
audit_receipt = {
    "assignment": "ML-09",
    "before_split_accuracy": round(float(acc_b), 4),
    "after_grouped_accuracy": round(float(np.mean(accs_after)), 4),
    "final_feature_count": len(final_features),
    "leakage_flagged_count": len(flagged_features),
    "safe_claim_language_used": True,
}
with open("work/outputs/ml09_validation_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit_receipt, f, indent=2)
print("Saved ML-09 audit summary to work/outputs/ml09_validation_audit.json")


Saved ML-09 audit summary to work/outputs/ml09_validation_audit.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.